In [ ]:
import itertools
import pandas as pd

# ==========================================
# 1. ENTRADA DE DATOS (Reemplaza con tus secuencias reales)
# ==========================================

# Secuencia del Adyuvante (CTB - UnitPROT: P01556)
adyuvante = "MIKLKFGVFFTVLLSSAYAHGTPQNITDLCAEYHNTQIYTLNDKIFSYTESLAGKREMAIITFKNGATFQVEVPGSQHIDSQKKAIERMKDTLRIAYLTEAKVEKLCVWNNKTPHAIAAISMAN"

# Epítopos individuales (reemplaza por tus secuencias de aminoácidos)
epitopos_B = ["GTQEQQIEQA", "GKNKDFSKAEET", "TQTQEKRHTTTKNTYAT", "ESYQKDAKELKN", "RVKEETKTSFH", "GMSQSNNPSKEE"]
epitopos_TCD8 = ["SAAEVLYQF", "AMNGIGIQV", "YSDNIAKEY", "FLPYGFNTDLL", "LPYGFNTDLLI", "VQVAIHTDTL"]
epitopos_TCD4 = ["GRTMHTFHTEGAGGG", "IDNFLEINNRVGSGA", "VSYNHLGSTNFESNS", "EFKELSNTAEKEGDK"]

# Enlazadores internos (dentro del bloque)
linker_B = "KK" #Linkers entre epítopos B
linker_TCD8 = "AAY" #Linke entre epítopos TCD8
linker_TCD4 = "GPGPG" #Linker entre epítopos TCD4

# Enlazadores estructurales (para unir los bloques y el adyuvante)
linker_adyuvante = "EAAAK" #Linker entre adyuvante y bloque
linker_bloques = "EAAAK"  # Linker entre bloques

# ==========================================
# 2. ENSAMBLAJE DE BLOQUES ESTÁTICOS
# ==========================================
# Aquí fijamos el orden interno de los epítopos en su respectivo bloque
bloque_B_seq = linker_B.join(epitopos_B)
bloque_CD8_seq = linker_TCD8.join(epitopos_TCD8)
bloque_CD4_seq = linker_TCD4.join(epitopos_TCD4)

# Diccionario para facilitar el manejo en el código
diccionario_bloques = {
    'B': bloque_B_seq,
    'CD8': bloque_CD8_seq,
    'CD4': bloque_CD4_seq
}

nombres_bloques = ['B', 'CD8', 'CD4']

# ==========================================
# 3. GENERACIÓN DE CONSTRUCTOS (FASTA y Datos para Excel)
# ==========================================
nombre_archivo_fasta = "constructos_racionales_V1_V12.fasta"
nombre_archivo_excel = "constructos_racionales_V1_V12.xlsx"

# Permutaciones de los 3 bloques (3! = 6 combinaciones)
combinaciones_bloques = list(itertools.permutations(nombres_bloques))

print("Iniciando diseño racional de vacunas (Macroe-estructuras)...")
print("-" * 50)

# Lista para almacenar los datos que irán al Excel
datos_excel = []
contador = 1

with open(nombre_archivo_fasta, "w") as f:
    for orden in combinaciones_bloques:
        # 1. Unir los bloques en el orden actual (ej. B -> CD8 -> CD4)
        secuencia_bloques_unidos = linker_bloques.join([diccionario_bloques[nombre] for nombre in orden])
        etiqueta_orden = "-".join(orden)

        # --- Arquitectura A: Adyuvante en N-terminal ---
        secuencia_N_term = adyuvante + linker_adyuvante + secuencia_bloques_unidos
        longitud_N = len(secuencia_N_term)
        nombre_V_N = f"V{contador}"

        # Escribir en FASTA
        header_N = f">{nombre_V_N} | Adyuvante:N-term | Orden_Bloques:{etiqueta_orden} | Longitud:{longitud_N}aa\n"
        f.write(header_N)
        f.write(f"{secuencia_N_term}\n")

        # Guardar datos para Excel
        datos_excel.append({
            "Variante": nombre_V_N,
            "Posición Adyuvante": "N-terminal",
            "Orden de Bloques": etiqueta_orden,
            "Longitud (aa)": longitud_N,
            "Antigenicidad (VaxiJen)": "",   # Columnas vacías para que tú las llenes
            "Alergenicidad (AllerTOP)": "",
            "Inestabilidad (ProtParam)": "",
            "Solubilidad": "",
            "Secuencia Completa": secuencia_N_term
        })
        print(f"Generado {nombre_V_N}: N-term | {etiqueta_orden}")
        contador += 1

        # --- Arquitectura B: Adyuvante en C-terminal ---
        secuencia_C_term = secuencia_bloques_unidos + linker_adyuvante + adyuvante
        longitud_C = len(secuencia_C_term)
        nombre_V_C = f"V{contador}"

        # Escribir en FASTA
        header_C = f">{nombre_V_C} | Adyuvante:C-term | Orden_Bloques:{etiqueta_orden} | Longitud:{longitud_C}aa\n"
        f.write(header_C)
        f.write(f"{secuencia_C_term}\n")

        # Guardar datos para Excel
        datos_excel.append({
            "Variante": nombre_V_C,
            "Posición Adyuvante": "C-terminal",
            "Orden de Bloques": etiqueta_orden,
            "Longitud (aa)": longitud_C,
            "Antigenicidad (VaxiJen)": "",
            "Alergenicidad (AllerTOP)": "",
            "Inestabilidad (ProtParam)": "",
            "Solubilidad": "",
            "Secuencia Completa": secuencia_C_term
        })
        print(f"Generado {nombre_V_C}: C-term | {etiqueta_orden}")
        contador += 1

# ==========================================
# 4. EXPORTAR A EXCEL
# ==========================================
# Convertir la lista de diccionarios en un DataFrame de Pandas
df = pd.DataFrame(datos_excel)

# Exportar a Excel
df.to_excel(nombre_archivo_excel, index=False)

print("-" * 50)
print(f"¡Proceso completado! Se generaron {contador - 1} secuencias.")
print(f"-> Archivo FASTA guardado como: '{nombre_archivo_fasta}'")
print(f"-> Archivo EXCEL guardado como: '{nombre_archivo_excel}'")

Iniciando diseño racional de vacunas (Macroe-estructuras)...
--------------------------------------------------
Generado V1: N-term | B-CD8-CD4
Generado V2: C-term | B-CD8-CD4
Generado V3: N-term | B-CD4-CD8
Generado V4: C-term | B-CD4-CD8
Generado V5: N-term | CD8-B-CD4
Generado V6: C-term | CD8-B-CD4
Generado V7: N-term | CD8-CD4-B
Generado V8: C-term | CD8-CD4-B
Generado V9: N-term | CD4-B-CD8
Generado V10: C-term | CD4-B-CD8
Generado V11: N-term | CD4-CD8-B
Generado V12: C-term | CD4-CD8-B
--------------------------------------------------
¡Proceso completado! Se generaron 12 secuencias.
-> Archivo FASTA guardado como: 'constructos_racionales_V1_V12.fasta'
-> Archivo EXCEL guardado como: 'constructos_racionales_V1_V12.xlsx'
